# 🥇 Gold Layer: Business Analytics, Warehouse Modeling & Comprehensive Exploratory Analytics

Welcome to **`gold.ipynb`**. This notebook handles **Data Engineering, Analytical Warehouse Modeling, Exploratory GroupBy Analytics, and the required AI Deliverable Export**:

### **What happens in this notebook?**
1. **Step 1: Load Clean Silver Dataset** (`data/processed/tech_news_clean.csv`).
2. **Step 2: Build Company Dimension (`dim_company`)** (1 row per company: HQ, industry, size, stock ticker).
3. **Step 3: Build Article Fact Table (`fct_article`)** (1 row per news article with foreign keys).
4. **Step 4: Build ARR Observation Fact Table (`fct_arr_observation`)** (Tracks financial observations with source lineage).
5. **Step 5: Compute Business Aggregations & Views**:
   - **Quarterly ARR Summary (`agg_company_quarterly_arr`)**: Tracks revenue growth per company per quarter.
   - **Latest Known ARR View (`view_company_latest_arr`)**: Current financial snapshot for each company.
6. **Step 6: Comprehensive Multi-Dimensional GroupBy Analytics** (Industry, Category, Author, Headquarters, Company Size).
7. **Step 7: Export Warehouse Tables** into `data/warehouse/` as standalone CSVs.
8. **Step 8: Export Required AI Dataset (`ai_articles_enriched.csv`)** as specified in Section 3 of `Task.txt`.

--- 
### **Step 1: Load Clean Silver Dataset**

In [16]:
import pandas as pd
import numpy as np
import os

df_silver = pd.read_csv('data/processed/tech_news_clean.csv')

# Enforce numeric types cleanly for aggregations
numeric_cols = ['published_year', 'published_quarter', 'published_month', 'word_count', 'revenue_usd_M', 'founded_year', 'employee_count', 'company_age']
for c in numeric_cols:
    if c in df_silver.columns:
        df_silver[c] = pd.to_numeric(df_silver[c], errors='coerce')

print(f"✅ Loaded Cleaned Silver Dataset: {df_silver.shape} (750 articles × 25 columns)")
display(df_silver[['article_id', 'company_name_clean', 'category_clean', 'published_date_clean', 'revenue_usd_M', 'company_size_category']].head(5))

✅ Loaded Cleaned Silver Dataset: (750, 25) (750 articles × 25 columns)


,article_id,company_name_clean,category_clean,published_date_clean,revenue_usd_M,company_size_category
0,ART0661,Airbnb,AI_ML,18-01-2024,7878.0,Medium
1,ART0118,Airbnb,AI_ML,27-02-2020,2600.0,Medium
2,ART0725,Airbnb,AI_ML,08-04-2024,NaN,Medium
3,ART0504,Airbnb,AI_ML,27-11-2023,7800.0,Medium
4,ART0271,Airbnb,AI_ML,31-10-2022,NaN,Medium


--- 
### **Step 2: Build Company Dimension Table (`dim_company`)**
- **Grain**: 1 row per company.
- **Primary Key**: `company_id` (`COMP001`, `COMP002`, ...).
- **Purpose**: Centralized master company directory containing firmographics (industry, headquarters, founded year, size).

In [17]:
# 1. Create unique company IDs
companies_list = sorted(df_silver['company_name_clean'].unique())
company_id_map = {name: f"COMP{i+1:03d}" for i, name in enumerate(companies_list)}

# 2. Aggregate to 1 row per company
dim_company = df_silver.groupby('company_name_clean').agg({
    'industry': 'first',
    'headquarters': 'first',
    'founded_year': 'first',
    'employee_count': 'first',
    'company_size_category': 'first',
    'is_public': 'first',
    'stock_ticker': 'first',
    'has_company_metadata': 'first'
}).reset_index()

dim_company.insert(0, 'company_id', dim_company['company_name_clean'].map(company_id_map))
dim_company.rename(columns={'company_name_clean': 'company_name'}, inplace=True)

print(f"✅ dim_company created: {dim_company.shape} (1 row per company)")
display(dim_company.head(6))

✅ dim_company created: (26, 10) (1 row per company)


,company_id,company_name,industry,headquarters,founded_year,employee_count,company_size_category,is_public,stock_ticker,has_company_metadata
0,COMP001,Airbnb,Data Analytics,"London, UK",1999.0,19967.0,Medium,False,NaN,True
1,COMP002,Amazon Web Services,SaaS,"Berlin, Germany",2015.0,27826.0,Medium,False,NaN,True
2,COMP003,Anthropic,FinTech,"San Francisco, CA",2006.0,43747.0,Large,False,NaN,True
3,COMP004,Cloudflare,Cybersecurity,"Austin, TX",1998.0,4422.0,Small,False,CLOU,True
4,COMP005,Cohere,NaN,NaN,NaN,NaN,Unknown,None,NaN,False
5,COMP006,Confluent,Cloud Computing,"London, UK",1995.0,3884.0,Small,False,NaN,True


--- 
### **Step 3: Build Article Fact Table (`fct_article`)**
- **Grain**: 1 row per news article.
- **Primary Key**: `article_id` | **Foreign Key**: `company_id`.
- **Purpose**: Full catalog of all news articles, author metadata, categories, and URLs.

In [18]:
fct_article = df_silver[[
    'article_id', 'original_index', 'company_name_clean', 'title', 
    'category', 'category_clean', 'author', 'published_date_clean', 
    'published_year', 'published_quarter', 'published_month', 'published_year_month', 
    'word_count', 'summary', 'url'
]].copy()

# Link to company dimension using foreign key company_id
fct_article.insert(2, 'company_id', fct_article['company_name_clean'].map(company_id_map))
fct_article.drop(columns=['company_name_clean'], inplace=True)

print(f"✅ fct_article created: {fct_article.shape} (1 row per article)")
display(fct_article.head(5))

✅ fct_article created: (750, 15) (1 row per article)


,article_id,original_index,company_id,title,category,category_clean,author,published_date_clean,published_year,published_quarter,published_month,published_year_month,word_count,summary,url
0,ART0661,660,COMP001,Airbnb Achieves Profitability Milestone,Artificial Intelligence,AI_ML,John Smith,18-01-2024,2024,1,1,2024-01,1885.0,Community contribution aims to accelerate inno...,https://technews.example.com/articles/661
1,ART0118,117,COMP001,Airbnb Reports Record Revenue Growth,AI & ML,AI_ML,Sam Wilson,27-02-2020,2020,1,2,2020-02,1068.0,International expansion strategy focuses on en...,https://technews.example.com/articles/118
2,ART0725,724,COMP001,Airbnb CEO Discusses Future of AI,AI/ML,AI_ML,Unknown,08-04-2024,2024,2,4,2024-04,1588.0,The company demonstrated significant advances ...,https://technews.example.com/articles/725
3,ART0504,503,COMP001,Airbnb Faces Regulatory Scrutiny,AI & ML,AI_ML,Unknown,27-11-2023,2023,4,11,2023-11,988.0,Platform improvements deliver enhanced perform...,https://technews.example.com/articles/504
4,ART0271,270,COMP001,Airbnb Partners with Major Enterprise Client,AI & ML,AI_ML,Unknown,31-10-2022,2022,4,10,2022-10,1910.0,Major partnership validates technology and ope...,https://technews.example.com/articles/271


--- 
### **Step 4: Build ARR Observation Fact Table (`fct_arr_observation`)**
- **Grain**: 1 row per valid financial observation.
- **Primary Key**: `observation_id` | **Foreign Keys**: `article_id`, `company_id`.
- **Lineage Rule**: Excludes missing/undisclosed records so financial modeling is never skewed by zeroes.

In [19]:
# Filter for valid revenue observations only
valid_arr_df = df_silver[df_silver['revenue_usd_M'].notnull()].copy()

fct_arr_observation = pd.DataFrame({
    'observation_id': [f"ARR{i+1:04d}" for i in range(len(valid_arr_df))],
    'article_id': valid_arr_df['article_id'].values,
    'company_id': valid_arr_df['company_name_clean'].map(company_id_map).values,
    'observation_date': valid_arr_df['published_date_clean'].values,
    'observation_year': valid_arr_df['published_year'].astype(int).values,
    'observation_quarter': valid_arr_df['published_quarter'].astype(int).values,
    'observation_year_month': valid_arr_df['published_year_month'].values,
    'arr_usd_M': valid_arr_df['revenue_usd_M'].astype(int).values,
    'arr_usd': (valid_arr_df['revenue_usd_M'].values * 1_000_000).astype(int)
})

print(f"✅ fct_arr_observation created: {fct_arr_observation.shape} (1 row per valid ARR observation)")
display(fct_arr_observation.head(5))

✅ fct_arr_observation created: (558, 9) (1 row per valid ARR observation)


,observation_id,article_id,company_id,observation_date,observation_year,observation_quarter,observation_year_month,arr_usd_M,arr_usd
0,ARR0001,ART0661,COMP001,18-01-2024,2024,1,2024-01,7878,7878000000
1,ARR0002,ART0118,COMP001,27-02-2020,2020,1,2020-02,2600,2600000000
2,ARR0003,ART0504,COMP001,27-11-2023,2023,4,2023-11,7800,7800000000
3,ARR0004,ART0398,COMP001,24-06-2022,2022,2,2022-06,5600,5600000000
4,ARR0005,ART0133,COMP001,04-04-2021,2021,2,2021-04,3800,3800000000


--- 
### **Step 5: Business Aggregations & Analytical Views**

#### **1. Quarterly ARR Summary (`agg_company_quarterly_arr`)**
Groups ARR observations by company and quarter to track financial progression over time.

In [20]:
# 1. Compute Quarterly Aggregations
agg_company_quarterly_arr = fct_arr_observation.groupby(['company_id', 'observation_year', 'observation_quarter']).agg(
    observations_count=('observation_id', 'count'),
    latest_arr_usd_M=('arr_usd_M', 'last'),
    avg_arr_usd_M=('arr_usd_M', 'mean'),
    max_arr_usd_M=('arr_usd_M', 'max'),
    min_arr_usd_M=('arr_usd_M', 'min')
).reset_index()
agg_company_quarterly_arr['avg_arr_usd_M'] = agg_company_quarterly_arr['avg_arr_usd_M'].round(1)

# Enrich with company name & industry
agg_company_quarterly_arr = pd.merge(agg_company_quarterly_arr, dim_company[['company_id', 'company_name', 'industry']], on='company_id', how='left')

# 2. Compute Latest Known ARR View (view_company_latest_arr)
view_company_latest_arr = fct_arr_observation.sort_values(by=['company_id', 'observation_year', 'observation_quarter']).groupby('company_id').agg(
    latest_observation_id=('observation_id', 'last'),
    latest_article_id=('article_id', 'last'),
    latest_observation_date=('observation_date', 'last'),
    latest_arr_usd_M=('arr_usd_M', 'last'),
    total_observations_recorded=('observation_id', 'count')
).reset_index()
view_company_latest_arr = pd.merge(dim_company[['company_id', 'company_name', 'industry', 'company_size_category', 'is_public']], view_company_latest_arr, on='company_id', how='left')

print("=== Sample Quarterly ARR Aggregation (Airbnb Growth) ===")
display(agg_company_quarterly_arr[agg_company_quarterly_arr['company_name'] == 'Airbnb'].head(6))

print("\n=== Latest Known ARR View (All Companies) ===")
display(view_company_latest_arr.head(6))

=== Sample Quarterly ARR Aggregation (Airbnb Growth) ===


,company_id,observation_year,observation_quarter,observations_count,latest_arr_usd_M,avg_arr_usd_M,max_arr_usd_M,min_arr_usd_M,company_name,industry
0,COMP001,2020,1,2,2600,2600.0,2600,2600,Airbnb,Data Analytics
1,COMP001,2020,4,1,3000,3000.0,3000,3000,Airbnb,Data Analytics
2,COMP001,2021,1,1,3400,3400.0,3400,3400,Airbnb,Data Analytics
3,COMP001,2021,2,1,3800,3800.0,3800,3800,Airbnb,Data Analytics
4,COMP001,2021,3,2,4200,4200.0,4200,4200,Airbnb,Data Analytics
5,COMP001,2022,2,1,5600,5600.0,5600,5600,Airbnb,Data Analytics



=== Latest Known ARR View (All Companies) ===


,company_id,company_name,industry,company_size_category,is_public,latest_observation_id,latest_article_id,latest_observation_date,latest_arr_usd_M,total_observations_recorded
0,COMP001,Airbnb,Data Analytics,Medium,False,ARR0007,ART0565,20-11-2024,8190,17
1,COMP002,Amazon Web Services,SaaS,Medium,False,ARR0026,ART0501,04-10-2024,102600,24
2,COMP003,Anthropic,FinTech,Large,False,ARR0058,ART0668,03-10-2024,1034,25
3,COMP004,Cloudflare,Cybersecurity,Small,False,ARR0091,ART0712,03-11-2024,1647,28
4,COMP005,Cohere,NaN,Unknown,None,ARR0098,ART0750,15-07-2024,135,6
5,COMP006,Confluent,Cloud Computing,Small,False,ARR0114,ART0555,27-10-2024,1156,33


--- 
### **Step 6: Comprehensive Multi-Dimensional GroupBy Analytics**

Here we perform deep-dive aggregations across **Industry, Article Category, Author, Headquarters, and Company Size**.

In [21]:
# =============================================================================
# Analysis 1: GroupBy Industry (Company Sectors Analysis)
# =============================================================================
industry_analysis = df_silver.groupby('industry').agg(
    total_articles=('article_id', 'count'),
    total_companies=('company_name_clean', 'nunique'),
    avg_revenue_usd_M=('revenue_usd_M', 'mean'),
    median_revenue_usd_M=('revenue_usd_M', 'median'),
    max_revenue_usd_M=('revenue_usd_M', 'max'),
    avg_word_count=('word_count', 'mean')
).reset_index().sort_values(by='total_articles', ascending=False)

industry_analysis['avg_revenue_usd_M'] = industry_analysis['avg_revenue_usd_M'].round(1)
industry_analysis['avg_word_count'] = industry_analysis['avg_word_count'].round(0)

print("📊 Analysis 1: Industry Performance & Revenue Breakdown")
display(industry_analysis)


# =============================================================================
# Analysis 2: GroupBy Article Category (Media Topic Breakdown)
# =============================================================================
category_analysis = df_silver.groupby('category_clean').agg(
    total_articles=('article_id', 'count'),
    distinct_companies=('company_name_clean', 'nunique'),
    avg_revenue_usd_M=('revenue_usd_M', 'mean'),
    avg_word_count=('word_count', 'mean'),
    earliest_publication=('published_year', 'min'),
    latest_publication=('published_year', 'max')
).reset_index().sort_values(by='total_articles', ascending=False)

category_analysis['avg_revenue_usd_M'] = category_analysis['avg_revenue_usd_M'].round(1)
category_analysis['avg_word_count'] = category_analysis['avg_word_count'].round(0)

print("\n📊 Analysis 2: Standardized Category Coverage & Article Metrics")
display(category_analysis)


# =============================================================================
# Analysis 3: GroupBy Author (Journalist & Coverage Productivity)
# =============================================================================
author_analysis = df_silver.groupby('author').agg(
    articles_published=('article_id', 'count'),
    companies_covered=('company_name_clean', 'nunique'),
    avg_article_word_count=('word_count', 'mean'),
    avg_reported_arr_usd_M=('revenue_usd_M', 'mean')
).reset_index().sort_values(by='articles_published', ascending=False)

author_analysis['avg_article_word_count'] = author_analysis['avg_article_word_count'].round(0)
author_analysis['avg_reported_arr_usd_M'] = author_analysis['avg_reported_arr_usd_M'].round(1)

print("\n📊 Analysis 3: Top Tech News Authors & Productivity")
display(author_analysis.head(10))


# =============================================================================
# Analysis 4: GroupBy Headquarters (Geographic Distribution)
# =============================================================================
hq_analysis = df_silver.groupby('headquarters').agg(
    companies_count=('company_name_clean', 'nunique'),
    articles_count=('article_id', 'count'),
    avg_employee_count=('employee_count', 'mean'),
    avg_company_revenue_usd_M=('revenue_usd_M', 'mean')
).reset_index().sort_values(by='companies_count', ascending=False)

hq_analysis['avg_employee_count'] = hq_analysis['avg_employee_count'].round(0)
hq_analysis['avg_company_revenue_usd_M'] = hq_analysis['avg_company_revenue_usd_M'].round(1)

print("\n📊 Analysis 4: Headquarters Geographic Clusters")
display(hq_analysis)


# =============================================================================
# Analysis 5: GroupBy Company Size Category (Small vs Medium vs Large)
# =============================================================================
size_analysis = df_silver.groupby('company_size_category').agg(
    distinct_companies=('company_name_clean', 'nunique'),
    total_articles=('article_id', 'count'),
    avg_employee_count=('employee_count', 'mean'),
    avg_revenue_usd_M=('revenue_usd_M', 'mean'),
    avg_company_age=('company_age', 'mean')
).reset_index().sort_values(by='distinct_companies', ascending=False)

size_analysis['avg_employee_count'] = size_analysis['avg_employee_count'].round(0)
size_analysis['avg_revenue_usd_M'] = size_analysis['avg_revenue_usd_M'].round(1)
size_analysis['avg_company_age'] = size_analysis['avg_company_age'].round(1)

print("\n📊 Analysis 5: Company Size Category Segmentation (<10k, 10k-30k, >30k)")
display(size_analysis)

📊 Analysis 1: Industry Performance & Revenue Breakdown


,industry,total_articles,total_companies,avg_revenue_usd_M,median_revenue_usd_M,max_revenue_usd_M,avg_word_count
3,Data Analytics,230,7,13864.3,1925.0,83850.0,1813.0
1,Cloud Computing,176,5,12722.6,1120.0,98880.0,1728.0
0,AI/ML,107,3,13666.4,7000.0,62100.0,1850.0
2,Cybersecurity,74,2,1075.1,1180.0,1647.0,1701.0
5,SaaS,73,2,35670.1,1870.0,102600.0,1674.0
4,FinTech,65,2,11853.0,743.0,37400.0,1807.0



📊 Analysis 2: Standardized Category Coverage & Article Metrics


,category_clean,total_articles,distinct_companies,avg_revenue_usd_M,avg_word_count,earliest_publication,latest_publication
0,AI_ML,161,26,9490.6,1717.0,2020,2024
3,Data_Analytics,131,26,20810.6,1819.0,2020,2024
5,FinTech,124,24,12919.3,1842.0,2020,2024
1,Cloud_Computing,119,24,15301.7,1746.0,2020,2024
2,Cybersecurity,104,23,14597.8,1775.0,2020,2024
4,Enterprise_Software,77,22,11208.1,1777.0,2020,2024
6,SaaS,34,17,9412.5,1892.0,2020,2024



📊 Analysis 3: Top Tech News Authors & Productivity


,author,articles_published,companies_covered,avg_article_word_count,avg_reported_arr_usd_M
4,Unknown,204,25,1738.0,12594.2
2,John Smith,146,24,1842.0,13557.6
0,Alex Johnson,140,26,1804.0,13055.0
3,Sam Wilson,137,24,1761.0,16749.0
1,Jane Doe,123,23,1790.0,13935.9



📊 Analysis 4: Headquarters Geographic Clusters


,headquarters,companies_count,articles_count,avg_employee_count,avg_company_revenue_usd_M
0,"Austin, TX",5,181,22025.0,15781.1
1,"Berlin, Germany",5,175,28656.0,17478.6
2,"London, UK",5,165,20384.0,21553.7
4,"San Francisco, CA",3,98,28353.0,8054.3
5,"Seattle, WA",2,72,28294.0,1751.4
3,"New York, NY",1,34,23471.0,308.4



📊 Analysis 5: Company Size Category Segmentation (<10k, 10k-30k, >30k)


,company_size_category,distinct_companies,total_articles,avg_employee_count,avg_revenue_usd_M,avg_company_age
1,Medium,12,394,20704.0,12646.2,12.8
0,Large,6,220,42311.0,15092.9,12.2
3,Unknown,5,25,NaN,110.2,NaN
2,Small,3,111,4620.0,18984.3,22.1


--- 
### **Step 7: Export Warehouse Relational Tables to `data/warehouse/`**
We save the relational warehouse tables as standalone CSV files.

In [22]:
os.makedirs('data/warehouse', exist_ok=True)

warehouse_tables = {
    'data/warehouse/dim_company.csv': dim_company,
    'data/warehouse/fct_article.csv': fct_article,
    'data/warehouse/fct_arr_observation.csv': fct_arr_observation,
    'data/warehouse/agg_company_quarterly_arr.csv': agg_company_quarterly_arr,
    'data/warehouse/view_company_latest_arr.csv': view_company_latest_arr
}

for fname, table_df in warehouse_tables.items():
    try:
        table_df.to_csv(fname, index=False)
        print(f"✅ Exported '{fname}' ({len(table_df)} rows)")
    except PermissionError:
        print(f"⚠️ '{fname}' is open in another app. Please close to overwrite.")

✅ Exported 'data/warehouse/dim_company.csv' (26 rows)
✅ Exported 'data/warehouse/fct_article.csv' (750 rows)
✅ Exported 'data/warehouse/fct_arr_observation.csv' (558 rows)
✅ Exported 'data/warehouse/agg_company_quarterly_arr.csv' (315 rows)
✅ Exported 'data/warehouse/view_company_latest_arr.csv' (26 rows)


--- 
### **Step 8: Export Required AI Dataset (`ai_articles_enriched.csv`)**

**As specified in Section 3 of `Task.txt`:**
1. **Category indicates AI/ML OR company industry indicates AI/ML**.
2. **Published between 2022 and 2024** (inclusive).
3. **Valid ARR > $50M USD** (`revenue_usd_M > 50`).
4. Export columns: `article_id`, `title`, `company_name`, `published_date`, `category`, `arr_usd`, `summary`, `url`, `industry`, `founded_year`, `headquarters`, `employee_count`, `is_public`, `stock_ticker`, `company_age`, `company_size_category`, `embedding`.

In [23]:
# 1. Filter articles according to exact task criteria
ai_mask = (
    ((df_silver['category_clean'] == 'AI_ML') | (df_silver['industry'] == 'AI/ML')) &
    (df_silver['published_year'] >= 2022) &
    (df_silver['published_year'] <= 2024) &
    (df_silver['revenue_usd_M'] > 50)
)

df_ai = df_silver[ai_mask].copy()
df_ai['company_name'] = df_ai['company_name_clean']
df_ai['published_date'] = df_ai['published_date_clean']
df_ai['arr_usd'] = (df_ai['revenue_usd_M'] * 1_000_000).astype(int)
df_ai['category'] = df_ai['category_clean']
df_ai['embedding'] = "[]"

ai_columns = [
    'article_id', 'title', 'company_name', 'published_date', 'category',
    'arr_usd', 'summary', 'url', 'industry', 'founded_year',
    'headquarters', 'employee_count', 'is_public', 'stock_ticker',
    'company_age', 'company_size_category', 'embedding'
]
ai_articles_enriched = df_ai[ai_columns].reset_index(drop=True)

# 2. Export to data/warehouse/ai_articles_enriched.csv
ai_export_path = 'data/warehouse/ai_articles_enriched.csv'
try:
    ai_articles_enriched.to_csv(ai_export_path, index=False)
    print(f"✅ Successfully exported '{ai_export_path}' ({len(ai_articles_enriched)} rows × {len(ai_articles_enriched.columns)} columns)")
except PermissionError:
    print(f"⚠️ '{ai_export_path}' is open in another app. Please close to overwrite.")

display(ai_articles_enriched.head(6))

✅ Successfully exported 'data/warehouse/ai_articles_enriched.csv' (124 rows × 17 columns)


,article_id,title,company_name,published_date,category,arr_usd,summary,url,industry,founded_year,headquarters,employee_count,is_public,stock_ticker,company_age,company_size_category,embedding
0,ART0661,Airbnb Achieves Profitability Milestone,Airbnb,18-01-2024,AI_ML,7878000000,Community contribution aims to accelerate inno...,https://technews.example.com/articles/661,Data Analytics,1999.0,"London, UK",19967.0,False,NaN,25.0,Medium,[]
1,ART0504,Airbnb Faces Regulatory Scrutiny,Airbnb,27-11-2023,AI_ML,7800000000,Platform improvements deliver enhanced perform...,https://technews.example.com/articles/504,Data Analytics,1999.0,"London, UK",19967.0,False,NaN,24.0,Medium,[]
2,ART0534,Anthropic Achieves Profitability Milestone,Anthropic,28-02-2023,AI_ML,320000000,Investors show strong confidence in the compan...,https://technews.example.com/articles/534,FinTech,2006.0,"San Francisco, CA",43747.0,False,NaN,17.0,Large,[]
3,ART0631,Anthropic Acquires Competitor for Undisclosed Sum,Anthropic,09-01-2024,AI_ML,602000000,Major partnership validates technology and ope...,https://technews.example.com/articles/631,FinTech,2006.0,"San Francisco, CA",43747.0,False,NaN,18.0,Large,[]
4,ART0249,Anthropic Reports Record Revenue Growth,Anthropic,17-05-2023,AI_ML,380000000,Strategic acquisition strengthens competitive ...,https://technews.example.com/articles/249,FinTech,2006.0,"San Francisco, CA",43747.0,False,NaN,17.0,Large,[]
5,ART0701,Anthropic Faces Regulatory Scrutiny,Anthropic,26-02-2024,AI_ML,602000000,Community contribution aims to accelerate inno...,https://technews.example.com/articles/701,FinTech,2006.0,"San Francisco, CA",43747.0,False,NaN,18.0,Large,[]
